# Assignment: Ham vs. Spam Classifier

**Goal:** Practice converting raw text into numerical features using a Bag of Words model and train a text classifier on `ham-spam.csv`.

**Tasks:**
1. Load the dataset and preprocess text (lowercase, remove punctuation and stop words)
2. Convert text to features with `CountVectorizer`
3. Split the data, train a classifier, and print scores plus a classification report
4. Predict labels for 3 of your own example messages
5. Answer the reflection questions as comments in the code

## Imports

In [ ]:
import re

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

## Step 1: Load the Dataset

Each row has a `label` (ham or spam) and the raw SMS message `text`.

In [ ]:
df = pd.read_csv("../data/ham-spam.csv")
print(f"Loaded {len(df)} messages")
df["label"].value_counts()

## Step 2: Lowercase the Text

So "Free" and "free" are treated as the same word.

In [ ]:
df["text"] = df["text"].astype(str).str.lower()

## Step 3: Remove Punctuation

Strip everything except letters, numbers, and spaces, then collapse any leftover extra whitespace.

In [ ]:
df["text"] = df["text"].apply(lambda t: re.sub(r"[^a-z0-9\s]", " ", t))
df["text"] = df["text"].apply(lambda t: re.sub(r"\s+", " ", t).strip())

## Step 4: Split into Train and Test Sets

`stratify=y` keeps the same ham/spam ratio in both the training and test splits. Stop words will be removed next, inside `CountVectorizer`.

In [ ]:
X_text = df["text"]
y = df["label"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

## Step 5: Convert Text to Bag-of-Words Features

We fit the vectorizer **only** on the training text to avoid data leakage, then reuse it to transform the test text. `stop_words='english'` removes common low-value words.

In [ ]:
vectorizer = CountVectorizer(stop_words="english")
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")
print(f"Training matrix shape: {X_train.shape}")

## Step 6: Train the Classifier

In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

## Step 7: Evaluate the Model

Print training/test accuracy and a full classification report (precision, recall, f1) on the test set.

In [ ]:
train_score = clf.score(X_train, y_train)
test_score = clf.score(X_test, y_test)

print(f"Training accuracy: {train_score:.4f}")
print(f"Test accuracy: {test_score:.4f}")
print()
print(classification_report(y_test, clf.predict(X_test)))

## Step 8: Test With Custom Examples

Write 3 of our own messages and see whether the trained model calls them ham or spam. New text must be transformed with the **same** vectorizer used during training.

In [ ]:
my_messages = [
    "Hey mom, dinner is ready when you get home tonight",
    "CONGRATULATIONS! You won a free iPhone! Click here to claim your prize now",
    "Meeting moved to 3pm tomorrow in conference room B",
]

for message in my_messages:
    features = vectorizer.transform([message])
    prediction = clf.predict(features)[0]
    print(f"  [{prediction.upper()}] {message}")

## Reflection Questions

Answered as comments below, per the assignment instructions.

In [ ]:
# Q: How well did your model perform?
# A: The model performed very well - about 98% test accuracy. Ham messages were
#    classified almost perfectly (high recall), while spam had slightly lower recall
#    (~83%), meaning some spam slipped through as ham.

# Q: Were any predictions surprising?
# A: No, all three custom examples matched what we expected. Casual personal
#    messages were labeled ham, and the obvious prize-scam wording was labeled spam.

# Q: What might help improve the model in future?
# A: Try TF-IDF weighting, n-grams (bigrams), a larger/more diverse dataset,
#    stemming or lemmatization for informal SMS spelling, and tuning the classifier
#    (e.g., MultinomialNB or class weights) to catch more spam without hurting ham recall.